In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np 
import copy
import random
import math
import os
import re
import json

# ================= 配置参数 =================
BOARD_SIZE = 15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "gomoku_model.pth"
BEST_LENGTH_MODEL_DIR = "best_length_models"
TRAIN_ITERATIONS = 1500  # 训练迭代次数 (实际使用需设为 1000+)
GAMES_PER_ITER = 1  # 每次迭代自我对弈局数
# modifies later
MCTS_SIMULATIONS = 30  # 每次决策模拟次数 (越大越强，越慢)

C_PUCT = 1.0  # MCTS 探索常数


In [ ]:

# ================= 1. 游戏逻辑引擎 =================
class GomokuGame:
    def __init__(self):
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)
        self.current_player = 1  # 1: 黑棋，-1: 白棋
        self.move_history = []
        self.winner = None

    def get_valid_moves(self):
        return list(zip(*np.where(self.board == 0)))

    def _line_properties(self, board, r, c, dr, dc, player):
        count = 1
        open_ends = 0

        # 向一个方向扩展
        i = 1
        while True:
            nr, nc = r + dr * i, c + dc * i
            if not (0 <= nr < BOARD_SIZE and 0 <= nc < BOARD_SIZE):
                break
            if board[nr, nc] == player:
                count += 1
            elif board[nr, nc] == 0:
                open_ends += 1
                break
            else:
                break
            i += 1

        # 另一个方向
        i = 1
        while True:
            nr, nc = r - dr * i, c - dc * i
            if not (0 <= nr < BOARD_SIZE and 0 <= nc < BOARD_SIZE):
                break
            if board[nr, nc] == player:
                count += 1
            elif board[nr, nc] == 0:
                open_ends += 1
                break
            else:
                break
            i += 1

        return count, open_ends

    def evaluate_move(self, move, player):
        r, c = move
        if self.board[r, c] != 0:
            return None

        tmp = self.board.copy()
        tmp[r, c] = player

        directions = [(1, 0), (0, 1), (1, 1), (1, -1)]
        best = None
        for dr, dc in directions:
            length, open_ends = self._line_properties(tmp, r, c, dr, dc, player)
            if length >= 5:
                return "five"
            if length == 4 and open_ends >= 1:
                best = "open_four"
            if length == 3 and open_ends == 2 and best != "open_four":
                best = "open_three"

        return best

    def find_critical_moves(self):
        cur = self.current_player
        opp = -cur

        win_moves = []
        block_moves = []
        open_four_moves = []
        open_three_moves = []

        for move in self.get_valid_moves():
            if self.evaluate_move(move, cur) == "five":
                win_moves.append(move)
            elif self.evaluate_move(move, opp) == "five":
                block_moves.append(move)
            elif self.evaluate_move(move, cur) == "open_four":
                open_four_moves.append(move)
            elif self.evaluate_move(move, cur) == "open_three":
                open_three_moves.append(move)

        return {
            "win": win_moves,
            "block": block_moves,
            "open_four": open_four_moves,
            "open_three": open_three_moves,
        }

    def get_priority_move(self):
        critical = self.find_critical_moves()
        if critical["win"]:
            return critical["win"][0]
        if critical["block"]:
            return critical["block"][0]
        if critical["open_four"]:
            return critical["open_four"][0]
        if critical["open_three"]:
            return critical["open_three"][0]
        return None

    def is_terminal(self):
        # 检查五子连珠
        directions = [(1, 0), (0, 1), (1, 1), (1, -1)]
        for r in range(BOARD_SIZE):
            for c in range(BOARD_SIZE):
                if self.board[r, c] == 0:
                    continue
                player = self.board[r, c]
                for dr, dc in directions:
                    count = 1
                    for i in range(1, 5):
                        nr, nc = r + dr * i, c + dc * i
                        if 0 <= nr < BOARD_SIZE and 0 <= nc < BOARD_SIZE and self.board[nr, nc] == player:
                            count += 1
                        else:
                            break
                    if count >= 5:
                        self.winner = player
                        return True
        # 检查平局
        if len(self.move_history) == BOARD_SIZE * BOARD_SIZE:
            self.winner = 0
            return True
        return False

    def make_move(self, move):
        r, c = move
        if self.board[r, c] != 0:
            return False
        self.board[r, c] = self.current_player
        self.move_history.append((r, c, self.current_player))
        self.is_terminal()
        self.current_player = -self.current_player
        return True

    def undo_move(self):
        if not self.move_history:
            return
        r, c, p = self.move_history.pop()
        self.board[r, c] = 0
        self.current_player = p
        self.winner = None

    def get_state_tensor(self):
        # 神经网络输入：2 个通道 (当前玩家棋子，对手棋子)
        # 始终从当前玩家视角构建输入
        own_board = (self.board == self.current_player).astype(float)
        opp_board = (self.board == -self.current_player).astype(float)
        state = np.stack([own_board, opp_board], axis=0)
        return torch.FloatTensor(state).unsqueeze(0).to(DEVICE)

    def get_reward(self):
        if self.winner is None:
            return 0
        if self.winner == 0:
            return 0
        return 1 if self.winner == self.current_player else -1 # 注意：这里是在游戏结束时相对于最后一步玩家的奖励

def print_board(board):
    print("    " + "".join(f"{chr(ord('a') + i):2}" for i in range(BOARD_SIZE)))
    for r in range(BOARD_SIZE):
        row_str = f"{r:2} "
        for c in range(BOARD_SIZE):
            if board[r, c] == 1:
                row_str += " ●"
            elif board[r, c] == -1:
                row_str += " ○"
            else:
                row_str += " ."
        print(row_str)

In [ ]:

# ================= 2. 神经网络 (Policy + Value) =================
class GomokuNet(nn.Module):
    def __init__(self):
        super().__init__()
        # U-Net 风格：编码器-解码器 + 跳跃连接
        self.enc1 = nn.Conv2d(2, 4, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(4, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # 瓶颈
        self.bottleneck = nn.Conv2d(8, 16, kernel_size=3, padding=1)

        # 解码器
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        self.dec2 = nn.Conv2d(16, 4, kernel_size=3, padding=1)  # 8 (from dec1) + 8 (skip) = 16
        self.final_up = nn.Upsample(size=(15, 15), mode='bilinear', align_corners=False)

        # 输出头
        self.policy_conv = nn.Conv2d(4, 1, kernel_size=1)

        # Value head
        self.value_fc1 = nn.Linear(16, 8)
        self.value_fc2 = nn.Linear(8, 1)

    def forward(self, x):
        # 编码器
        e1 = F.leaky_relu(self.enc1(x))
        e1 = F.leaky_relu(self.enc2(e1))
        p = self.pool(e1)

        # 瓶颈
        b = F.leaky_relu(self.bottleneck(p))

        # 价值头（从瓶颈）
        b_global = F.adaptive_avg_pool2d(b, (1, 1)).view(-1, 16)
        v = F.leaky_relu(self.value_fc1(b_global))
        v = torch.tanh(self.value_fc2(v))

        # 解码器 + 跳跃连接
        u = self.up(b)
        u = F.leaky_relu(self.dec1(u))
        u = torch.cat([u, e1[:, :, :14, :14]], dim=1)  # 跳跃连接，裁剪e1到14x14匹配u
        u = F.leaky_relu(self.dec2(u))
        u = self.final_up(u)  # 上采样到15x15

        # 策略头
        p_out = self.policy_conv(u)
        p_out = p_out.view(-1, BOARD_SIZE * BOARD_SIZE)
        p_out = F.softmax(p_out, dim=1)

        return p_out, v

In [ ]:

# ================= 3. MCTS 节点与搜索 =================
class MCTSNode:
    def __init__(self, parent=None, move=None):
        self.parent = parent
        self.move = move
        self.children = {}
        self.n = 0  # 访问次数
        self.w = 0  # 累计价值
        self.p = 0  # 先验概率 (由 NN 提供)
        self.q = 0  # 平均价值 (W/N)

    def is_expanded(self):
        return len(self.children) > 0

    def best_child(self, c_puct=1.0):
        best_score = -float('inf')
        best_node = None
        for move, node in self.children.items():
            # UCT 公式：Q + P * sqrt(N_parent) / (1 + N_child)
            uct = node.q + c_puct * node.p * math.sqrt(self.n) / (1 + node.n)
            if uct > best_score:
                best_score = uct
                best_node = node
        return best_node

class MCTS:
    def __init__(self, net, c_puct=1.0, n_simulations=50):
        self.net = net
        self.c_puct = c_puct
        self.n_simulations = n_simulations
        self.root = None

    def search(self, game):
        self.root = MCTSNode()
        for _ in range(self.n_simulations):
            game_copy = copy.deepcopy(game)
            node = self.root
            path = []
            
            # 1. Selection
            while node.is_expanded() and not game_copy.is_terminal():
                node = node.best_child(self.c_puct)
                if node.move:
                    game_copy.make_move(node.move)
                path.append(node)
            
            # 2. Expansion & Evaluation
            if not game_copy.is_terminal():
                state = game_copy.get_state_tensor()
                with torch.no_grad():
                    policy, value = self.net(state)
                
                policy = policy.cpu().numpy()[0]
                value = value.cpu().numpy()[0][0]
                
                # 关键走法：连五/冲四/活三优先
                critical = game_copy.find_critical_moves()
                if critical["win"]:
                    valid_moves = critical["win"]
                elif critical["block"]:
                    valid_moves = critical["block"]
                elif critical["open_four"]:
                    valid_moves = critical["open_four"]
                elif critical["open_three"]:
                    valid_moves = critical["open_three"]
                else:
                    # 仅保留第一步固定中间；第二步强制在中间5x5范围内；从第3步起，从对方上一步和自己上一步邻域搜索
                    if len(game_copy.move_history) == 1:
                        # 第二步：中间5x5范围 (5-9, 5-9)
                        valid_moves = [move for move in game_copy.get_valid_moves() if 5 <= move[0] <= 9 and 5 <= move[1] <= 9]
                    elif len(game_copy.move_history) >= 2:
                        last_r, last_c, _ = game_copy.move_history[-1]
                        prev_r, prev_c, _ = game_copy.move_history[-2]
                        focused = set()
                        radius = 2
                        for center_r, center_c in [(last_r, last_c), (prev_r, prev_c)]:
                            for dr in range(-radius, radius + 1):
                                for dc in range(-radius, radius + 1):
                                    nr, nc = center_r + dr, center_c + dc
                                    if 0 <= nr < BOARD_SIZE and 0 <= nc < BOARD_SIZE and game_copy.board[nr, nc] == 0:
                                        focused.add((nr, nc))
                        valid_moves = list(focused) if focused else game_copy.get_valid_moves()
                    else:
                        valid_moves = game_copy.get_valid_moves()

                # 过滤非法移动的概率
                policy_vec = np.zeros(BOARD_SIZE * BOARD_SIZE)
                for i, (r, c) in enumerate(valid_moves):
                    idx = r * BOARD_SIZE + c
                    policy_vec[idx] = policy[idx]
                
                # 归一化
                s = np.sum(policy_vec)
                if s > 0:
                    policy_vec /= s
                
                for r, c in valid_moves:
                    idx = r * BOARD_SIZE + c
                    child = MCTSNode(parent=node, move=(r, c))
                    child.p = policy_vec[idx]
                    node.children[(r, c)] = child
                
                # 如果刚扩展，直接反向传播评估值
                if not node.children:
                    # 没有合法移动 (平局或结束)
                    pass 
                else:
                    # 这里简化处理：如果是叶子节点，使用 NN 的 Value
                    # 在 AlphaZero 中，叶子节点如果是终止状态用真实奖励，否则用 V(s)
                    # 为了代码简洁，这里统一用 V(s) 进行第一次备份，真实结果在最终决定时修正
                    pass 
            else:
                # 游戏结束
                value = 1.0 if game_copy.winner == game_copy.current_player else -1.0 # 注意视角
                # 修正：game_copy.current_player 是下一个要下的人，如果游戏结束，winner 是上一个下的人
                # 所以如果 winner == 上一个玩家，对当前节点 (上一个玩家的决策) 来说是胜利
                # 这里逻辑较绕，简化为：如果是终止状态，根据 winner 赋值
                if game_copy.winner == 0: value = 0
                elif game_copy.winner == game_copy.move_history[-1][2]: value = 1
                else: value = -1

            # 3. Back propagation
            for node in reversed(path):
                node.n += 1
                node.w += value
                node.q = node.w / node.n
                value = -value # 零和博弈，价值取反

    def get_action_distribution(self, game, temperature=1.05): # keeps random going lower
        # 优先处理游戏本身给出的关键走法（五/堵/冲四/活三），人机对战也会走这一条
        priority_move = game.get_priority_move()
        if priority_move is not None:
            dist = np.zeros(BOARD_SIZE * BOARD_SIZE)
            dist[priority_move[0] * BOARD_SIZE + priority_move[1]] = 1.0
            return dist, priority_move

        self.search(game)
        # 根据根节点子节点的访问次数生成策略
        counts = np.zeros(BOARD_SIZE * BOARD_SIZE)
        for move, node in self.root.children.items():
            idx = move[0] * BOARD_SIZE + move[1]
            counts[idx] = node.n
        
        if temperature == 0:
            # 选择访问次数最多的
            best_move = max(self.root.children, key=lambda k: self.root.children[k].n)
            dist = np.zeros(BOARD_SIZE * BOARD_SIZE)
            dist[best_move[0]*BOARD_SIZE + best_move[1]] = 1.0
            return dist, best_move
        else:
            # 应用温度参数
            counts = counts ** (1 / temperature)
            counts_sum = np.sum(counts)
            if counts_sum == 0:
                # 如果没有子节点 (极端情况)，随机选
                valid = game.get_valid_moves()
                if not valid: return None, None
                move = random.choice(valid)
                dist = np.zeros(BOARD_SIZE * BOARD_SIZE)
                dist[move[0]*BOARD_SIZE + move[1]] = 1.0
                return dist, move
            
            probs = counts / counts_sum
            # 采样
            move_idx = np.random.choice(len(probs), p=probs)
            move = (move_idx // BOARD_SIZE, move_idx % BOARD_SIZE)
            return probs, move


In [ ]:
from time import sleep

class Trainer:
    def __init__(self):
        self.net = GomokuNet().to(DEVICE)
        self.optimizer = optim.Adam(self.net.parameters(), lr=1e-3)
        self.mcts = MCTS(self.net, n_simulations=MCTS_SIMULATIONS)
        self.replay_buffer = []
        # 加载或初始化最长游戏步数
        if os.path.exists("config.json"):
            with open("config.json", "r") as f:
                config = json.load(f)
                self.longest_game_length = config.get("max_moves", 0)
        else:
            self.longest_game_length = 20

    def self_play(self):
        game = GomokuGame()
        states = []
        mcts_policies = []

        while not game.is_terminal():
            if not game.move_history:
                # 第一步固定在中间
                move = (7, 7)
                action_probs = np.zeros(BOARD_SIZE * BOARD_SIZE,dtype="float")
                action_probs[7 * BOARD_SIZE + 7] = 1.0
            else:
                action_probs, move = self.mcts.get_action_distribution(game)

            if move is None:
                break

            state = game.get_state_tensor()
            states.append(state)
            mcts_policies.append(action_probs)
            game.make_move(move)

        # 计算最终奖励 (z)
        winner = game.winner
        total_steps = len(game.move_history)
        # 长局奖励系数：步数越多，胜利奖励更高（最大2倍），失败惩罚更低（最小0.5倍）
        if total_steps <= 20:
            length_factor = 1.0
        elif total_steps <= 60:
            length_factor = 1.0 + (total_steps - 20) / 80.0  # 1.0 到 1.5
        else:
            length_factor = 1.5 + min((total_steps - 60) / 120.0, 0.5)  # 1.5 到 2.0

        rewards = []
        for i in range(len(states)):
            # 对于每一步，如果最终赢家是该步的玩家，奖励为 1 * length_factor，否则 -1 * loss_factor
            step_player = 1 if i % 2 == 0 else -1
            if winner == step_player:
                rewards.append(1.0 * length_factor)
            elif winner == 0:
                rewards.append(0.0)
            else:
                loss_factor = max(0.5, 1.0 - (length_factor - 1.0) * 0.8)
                rewards.append(-1.0 * loss_factor)

        return states, mcts_policies, rewards, game

    def train_step(self):
        self.net.train()
        states, policies, rewards = [], [], []

        print(f"Generating {GAMES_PER_ITER} self-play games...")
        games = []
        for i in range(GAMES_PER_ITER):
            sleep(0.7)
            print(f"Game {i+1}  ",end="\r")
            s, p, r, g = self.self_play()
            states.extend(s)
            policies.extend(p)
            rewards.extend(r)
            games.append(g)

        # 保存当前迭代模型
        torch.save(self.net.state_dict(), MODEL_PATH)

        # 记录并保存棋局步数最长的模型
        if games:
            longest_game = max(games, key=lambda g: len(g.move_history))
            longest_moves = len(longest_game.move_history)
            if longest_moves > self.longest_game_length:
                self.longest_game_length = longest_moves
                os.makedirs(BEST_LENGTH_MODEL_DIR, exist_ok=True)
                best_path = os.path.join(
                    BEST_LENGTH_MODEL_DIR,
                    f"gomoku_model_max_moves_{longest_moves}.pth"
                )
                torch.save(self.net.state_dict(), best_path)
                # 保存到config.json
                with open("config.json", "w") as f:
                    json.dump({"max_moves": self.longest_game_length}, f)
                print(f"New longest game ({longest_moves} 步)，模型已保存：{best_path}")

        if len(states) == 0:
            return

        # 输出一个示例棋局
        if games:
            print("示例棋局最终状态:")
            print_board(games[-1].board)

            print("下棋顺序:")
            for step, (r, c, p) in enumerate(games[-1].move_history):
                player = "黑" if p == 1 else "白"
                print(f"第{step+1}步: {player} ({r}{chr(ord('a') + c)})")
            if games[-1].winner == 0:
                print("结果: 平局")
            elif games[-1].winner == 1:
                print("结果: 黑棋胜")
            else:
                print("结果: 白棋胜")

        # 应用对称增强，避免重复训练对称局面
        """ states, policies, rewards = apply_symmetry_augmentation(
            states, policies, rewards
        ) """

        # 构建 Tensor
        state_tensor = torch.cat(states, dim=0).to(DEVICE)
        policy_tensor = torch.FloatTensor(np.array(policies)).to(DEVICE)
        value_tensor = torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(DEVICE)

        # 训练
        self.optimizer.zero_grad()
        pred_policy, pred_value = self.net(state_tensor)

        # Loss = Value MSE + Policy CrossEntropy
        loss_value = F.mse_loss(pred_value, value_tensor)
        loss_policy = torch.mean(
            torch.sum(-policy_tensor * torch.log(pred_policy + 1e-6), dim=1)
        )
        loss = loss_value + loss_policy

        loss.backward()
        self.optimizer.step()

        print(
            f"Loss: {loss.item():.4f} (Val: {loss_value.item():.4f}, Pol: {loss_policy.item():.4f})"
        )

        print(f"Model saved to {MODEL_PATH}")

    def load_model(self):
        if os.path.exists(MODEL_PATH):
            self.net.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
            print("Model loaded.")
        else:
            print("No model found, starting with random weights.")
        self.net.eval()

In [ ]:

# ================= 5. 人机对战接口 =================
def human_vs_ai():
    trainer = Trainer()
    trainer.load_model()
    # 对战时使用训练好的模型，不再更新权重
    trainer.net.eval() 
    mcts = MCTS(trainer.net, n_simulations=MCTS_SIMULATIONS)
    
    game = GomokuGame()
    print("=== 五子棋人机对战 (黑棋 ● 先手) ===")
    print("输入格式：纵坐标数字 + 横坐标字母 (例如：7h)")
    
    # 让玩家选择先后手
    human_color = 1
    ai_color = -1
    choice = input("选择先手 (1) 或 后手 (-1): \n第一步尽量走7h, 第二步尽量r>=c>=0")
    if choice == '-1':
        human_color = -1
        ai_color = 1
        print("AI 执黑先行。")
        # AI 先下
        # action_probs, move = mcts.get_action_distribution(game, temperature=0)
        move=(7,7)
        game.make_move(move)
        print(f"AI 落子：{move[0]}{chr(ord('a') + move[1])}")
    else:
        print("你执黑先行。")

    print_board(game.board)

    while not game.is_terminal():
        if game.current_player == human_color:
            try:
                inp = input("(q to exit)你的回合 (纵坐标数字 + 横坐标字母): ").strip()
                if inp=="q":
                    print("Pressed Q. Exiting...\n")
                    return
                match = re.match(r'(\d+)([a-o])', inp)
                if not match:
                    print("输入格式错误，请重试。")
                    continue
                r = int(match.group(1))
                c = ord(match.group(2)) - ord('a')
                if not (0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE):
                    print("坐标超出范围，请重试。")
                    continue
                if not game.make_move((r, c)):
                    print("无效落子，请重试。")
                    continue
            except Exception as e:
                print(f"输入错误：{e}")
                continue
        else:
            print("AI 思考中...")
            action_probs, move = mcts.get_action_distribution(game, temperature=0)
            if move:
                game.make_move(move)
                print(f"AI 落子：{move[0]}{chr(ord('a') + move[1])}")
            else:
                print("AI 认输 (无合法移动)")
                break
        
        print_board(game.board)

    if game.winner == 0:
        print("平局！")
    elif game.winner == human_color:
        print("恭喜你，你赢了！")
    else:
        print("AI 赢了！")


In [71]:

# ================= 主程序入口 =================
from time import sleep


if __name__ == "__main__":
    choice = input("1. 训练 AI (自我对弈)""2. 人机对战""请选择 (1/2): ")
    
    if choice == '1':
        trainer = Trainer()
        # 如果有旧模型则加载，否则从头开始
        if os.path.exists(MODEL_PATH):
            trainer.net.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
            print("加载了现有模型继续训练。")
        
        for i in range(TRAIN_ITERATIONS):
            print(f"\n--- 训练迭代 {i+1}/{TRAIN_ITERATIONS} ---")
            
            MCTS_SIMULATIONS=random.randrange(150,200)
            print(f"starting next | MCTS Sims: {MCTS_SIMULATIONS}")
            trainer.train_step()
    elif choice == '2':
        MCTS_SIMULATIONS=200
        human_vs_ai()
    else:
        print("无效选择")

示例棋局最终状态:
    a b c d e f g h i j k l m n o 
 0  . . . . . . . . . . . . . . .
 1  . . . . . . . . . . . . . . .
 2  . . . . . . . . . . . . . . .
 3  . . . . . . . . . . . . . . .
 4  . . . . . . . . . . . . . . .
 5  . . . . . ○ . . . . . ○ . . .
 6  . . . . . . . . . . . ● . . .
 7  . . . . . . . ● ○ . . ● . ○ .
 8  . . . . . . . . . ○ . ● . . .
 9  . . . . . . . . ● . ○ ● . . .
10  . . . . . . . . . . . ● . . .
11  . . . . . . . . . . . . . . .
12  . . . . . . . . . . . . . . .
13  . . . . . . . . . . . . . . .
14  . . . . . . . . . . . . . . .
下棋顺序:
第1步: 黑 (7h)
第2步: 白 (5f)
第3步: 黑 (9i)
第4步: 白 (8j)
第5步: 黑 (8l)
第6步: 白 (7n)
第7步: 黑 (9l)
第8步: 白 (9k)
第9步: 黑 (7l)
第10步: 白 (7i)
第11步: 黑 (6l)
第12步: 白 (5l)
第13步: 黑 (10l)
结果: 黑棋胜
Loss: 6.1962 (Val: 0.9939, Pol: 5.2022)
Model saved to gomoku_model.pth

--- 训练迭代 46/1500 ---
starting next | MCTS Sims: 180
Generating 1 self-play games...
示例棋局最终状态:
    a b c d e f g h i j k l m n o 
 0  . . . . . . . . . . . . . . .
 1  . . . . . . . . . . . . . . .


KeyboardInterrupt: 